# Wrapper Methods for Obesity Classification – Solution

**Short name (GitHub):** `WrapFS_Obes`  
**Lab source:** Codecademy *Wrapper Methods* (UCI obesity eating-habits survey)  
**Language:** Python (pandas + scikit-learn + mlxtend)

Attempt **`WrapFS_Obes_Practice_Skeleton.ipynb`** first. This notebook is the worked key plus alternates, extra practice, and a simulation.

Companion files: `WrapFS_Obes_Cheatsheet.docx`, `WrapFS_Obes_Reusable_Template.ipynb`, `wrapfs_obes_flowchart.png`, `WrapFS_Obes.py`, `WrapFS_Obes_1Page_Summary_Report.docx`.

### Headline numbers (`cv=0`, in-sample)
| Model | k | Accuracy |
|-------|---|----------|
| All 18 features | 18 | **0.766** |
| SFS | 6 | **0.769** |
| SFS | 9 | **0.784** |
| SBS / SBFS | 7 | **0.764** |
| RFE (scaled) | 6 | **0.758** |
| RFE (scaled) | 8 | **0.768** |

Core items that almost every wrapper keeps: **Age**, **family_history_with_overweight**, **FAVC**, **CAEC**, **SCC**.


## Inline cheat-sheet

See **`WrapFS_Obes_Cheatsheet.docx`**.

| Item | Code / rule |
|------|-------------|
| SFS | add the feature that most improves accuracy |
| SBS | drop the feature that least hurts accuracy |
| Floating | after add/drop, try the opposite move if it helps |
| RFE | drop smallest `\|coef\|` after each refit; scale first |
| `cv=0` | lesson default = resubstitution score |
| sklearn SFS | `SequentialFeatureSelector(..., direction="forward", cv=None)` |

**Flow:** load → baseline → SFS → SBS → scale → RFE → compare → practice → simulate.


## 0. Packages


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE, SequentialFeatureSelector as SkSFS
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from mlxtend.plotting import plot_sequential_feature_selection as plot_sfs

%matplotlib inline
np.set_printoptions(precision=4, suppress=True)
print("Libraries loaded")


## 1. Load and inspect


In [ ]:
obesity = pd.read_csv("data/obesity.csv")
print(obesity.shape)
display(obesity.head())
print(obesity["NObeyesdad"].value_counts())
print("obesity rate", obesity["NObeyesdad"].mean())
print(obesity.dtypes)


### 1.2 Split X / y


In [ ]:
X = obesity.drop(columns=["NObeyesdad"])
y = obesity["NObeyesdad"]
print("X shape", X.shape, "y mean", float(y.mean()))
print(list(X.columns))


## 2. Baseline logistic regression


In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X, y)
full_acc = lr.score(X, y)
print("all-18 in-sample accuracy:", full_acc)
# Expected ≈ 0.766


## 3. Sequential Forward Selection


In [ ]:
sfs = SFS(
    lr,
    k_features=9,
    forward=True,
    floating=False,
    scoring="accuracy",
    cv=0,
)
sfs.fit(X, y)
print(sfs.subsets_[9])
print("names:", sfs.subsets_[9]["feature_names"])
print("score:", sfs.subsets_[9]["avg_score"])
# Expected names include Age, family_history, FAVC, CAEC, SCC, FAF, Gender, Bike, Walking
# Expected score ≈ 0.784  (beats all-18)


In [ ]:
plot_sfs(sfs.get_metric_dict())
plt.title("SFS accuracy vs number of features (k=9)")
plt.tight_layout()
plt.show()


### 3.4 Published alternate k=6


In [ ]:
sfs6 = SFS(lr, k_features=6, forward=True, floating=False, scoring="accuracy", cv=0)
sfs6.fit(X, y)
print("SFS-6 names:", sfs6.subsets_[6]["feature_names"])
print("SFS-6 score:", sfs6.subsets_[6]["avg_score"])
# Expected: Age, family_history_with_overweight, FAVC, CAEC, SCC, FAF
# Expected score ≈ 0.769


## 4. Sequential Backward Selection


In [ ]:
sbs = SFS(lr, k_features=7, forward=False, floating=False, scoring="accuracy", cv=0)
sbs.fit(X, y)
print(sbs.subsets_[7])
print("SBS-7 names:", sbs.subsets_[7]["feature_names"])
print("SBS-7 score:", sbs.subsets_[7]["avg_score"])
# Expected ≈ 0.764 — slightly *below* the full model (greedy drop is not guaranteed to improve)
plot_sfs(sbs.get_metric_dict())
plt.title("SBS accuracy vs number of features (k=7)")
plt.tight_layout()
plt.show()


### 4.3 Floating (SBFS)


In [ ]:
sbfs = SFS(lr, k_features=7, forward=False, floating=True, scoring="accuracy", cv=0)
sbfs.fit(X, y)
print("SBFS-7 names:", sbfs.subsets_[7]["feature_names"])
print("SBFS-7 score:", sbfs.subsets_[7]["avg_score"])
# On this table SBFS-7 matches SBS-7.


## 5. Recursive Feature Elimination


In [ ]:
features = X.columns
X_scaled = pd.DataFrame(StandardScaler().fit_transform(X), columns=features)

rfe = RFE(estimator=lr, n_features_to_select=8)
rfe.fit(X_scaled, y)
rfe_features = [f for (f, support) in zip(features, rfe.support_) if support]
print("RFE-8 names:", rfe_features)
print("RFE-8 ranking:", rfe.ranking_)
print("RFE-8 score:", rfe.score(X_scaled, y))
# Expected ≈ 0.768
# names: Age, family_history, FAVC, FCVC, CAEC, SCC, Automobile, Walking

rfe6 = RFE(estimator=lr, n_features_to_select=6)
rfe6.fit(X_scaled, y)
rfe6_features = [f for (f, support) in zip(features, rfe6.support_) if support]
print("RFE-6 names:", rfe6_features)
print("RFE-6 score:", rfe6.score(X_scaled, y))
# Expected ≈ 0.758
# names: Age, family_history, FAVC, CAEC, SCC, Automobile


## 6. Compare the three wrappers


In [ ]:
rows = [
    ("All features", 18, full_acc, tuple(features)),
    ("SFS", 9, sfs.subsets_[9]["avg_score"], sfs.subsets_[9]["feature_names"]),
    ("SFS", 6, sfs6.subsets_[6]["avg_score"], sfs6.subsets_[6]["feature_names"]),
    ("SBS", 7, sbs.subsets_[7]["avg_score"], sbs.subsets_[7]["feature_names"]),
    ("RFE", 8, rfe.score(X_scaled, y), tuple(rfe_features)),
    ("RFE", 6, rfe6.score(X_scaled, y), tuple(rfe6_features)),
]
cmp = pd.DataFrame(rows, columns=["method", "k", "accuracy", "features"])
display(cmp)

sets = {
    "SFS6": set(sfs6.subsets_[6]["feature_names"]),
    "SFS9": set(sfs.subsets_[9]["feature_names"]),
    "SBS7": set(sbs.subsets_[7]["feature_names"]),
    "RFE6": set(rfe6_features),
    "RFE8": set(rfe_features),
}
core = set.intersection(*sets.values())
print("in EVERY subset:", sorted(core))
print("in at least 4 of 5:", sorted(f for f in set().union(*sets.values())
                                   if sum(f in s for s in sets.values()) >= 4))


## 7. Alternate code

### 7.1 sklearn SequentialFeatureSelector


In [ ]:
sk_fwd = SkSFS(lr, n_features_to_select=6, direction="forward", scoring="accuracy", cv=None)
sk_fwd.fit(X, y)
print("sklearn SFS-6:", list(X.columns[sk_fwd.get_support()]))

sk_bwd = SkSFS(lr, n_features_to_select=7, direction="backward", scoring="accuracy", cv=None)
sk_bwd.fit(X, y)
print("sklearn SBS-7:", list(X.columns[sk_bwd.get_support()]))
# Search path can differ from mlxtend; the scientific claim is the same: a short card ≈ full card.


### 7.2 L1-logistic (embedded)


In [ ]:
Xs = StandardScaler().fit_transform(X)
l1 = LogisticRegression(penalty="l1", solver="liblinear", C=0.08, max_iter=1000)
l1.fit(Xs, y)
kept = [f for f, c in zip(features, l1.coef_.ravel()) if abs(c) > 1e-8]
print("L1 kept (C=0.08):", kept)
print("L1 acc:", l1.score(Xs, y))
print("nonzero coefs:")
print(pd.Series(l1.coef_.ravel(), index=features)[lambda s: s.abs() > 1e-8].sort_values())


### 7.3 Mutual-information filter


In [ ]:
mi = mutual_info_classif(X, y, random_state=0)
mi_s = pd.Series(mi, index=features).sort_values(ascending=False)
print(mi_s.round(4))
print("MI top-6:", list(mi_s.head(6).index))
# Filter ignores the LR decision surface; wrappers optimize that surface directly.


## 8. More practice

### 8.1 Hold-out (honest) accuracy


In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, random_state=0, stratify=y)
cols9 = list(sfs.subsets_[9]["feature_names"])
acc18 = LogisticRegression(max_iter=1000).fit(Xtr, ytr).score(Xte, yte)
acc9 = LogisticRegression(max_iter=1000).fit(Xtr[cols9], ytr).score(Xte[cols9], yte)
print(f"hold-out all-18={acc18:.4f}  SFS-9={acc9:.4f}")
# In-sample SFS *gain* often shrinks (or vanishes) on a true hold-out. That is the point of Task 8.1.


### 8.2 Drop one transport dummy


In [ ]:
X2 = X.drop(columns=["Public_Transportation"])
sfs_ref = SFS(LogisticRegression(max_iter=1000), k_features=6,
              forward=True, floating=False, scoring="accuracy", cv=0)
sfs_ref.fit(X2, y)
print("SFS-6 after dropping Public_Transportation:", sfs_ref.subsets_[6]["feature_names"])
print("score:", sfs_ref.subsets_[6]["avg_score"])


### 8.3 Sweep k


In [ ]:
ks, scores = [], []
for k in range(2, 13):
    tmp = SFS(LogisticRegression(max_iter=1000), k_features=k,
              forward=True, floating=False, scoring="accuracy", cv=0)
    tmp.fit(X, y)
    ks.append(k)
    scores.append(tmp.subsets_[k]["avg_score"])
    print(k, round(scores[-1], 4), tmp.subsets_[k]["feature_names"])

plt.plot(ks, scores, marker="o")
plt.axhline(full_acc, ls="--", color="gray", label="all-18")
plt.xlabel("k_features"); plt.ylabel("in-sample acc")
plt.title("SFS accuracy by subset size")
plt.legend(frameon=False)
plt.tight_layout()
plt.show()
# Curve flattens near k=8–10 at ≈0.7835; k=6 already matches the full model.


## 9. Simulation

Edit the knobs. Each replication: optional subsample, optional label flips, optional junk columns, then SFS-k in-sample + hold-out LR on the chosen columns.


In [ ]:
# --- knobs ---
K_TARGET = 6
N_SUB = 800
FLIP_P = 0.00
N_JUNK = 0
N_REPS = 8
SEED = 7
# -------------

rng = np.random.default_rng(SEED)
ins, outs = [], []
for r in range(N_REPS):
    idx = rng.choice(len(X), size=(N_SUB or len(X)), replace=False)
    Xb = X.iloc[idx].reset_index(drop=True).copy()
    yb = y.iloc[idx].reset_index(drop=True).to_numpy().copy()
    if FLIP_P > 0:
        m = rng.random(len(yb)) < FLIP_P
        yb[m] = 1 - yb[m]
    if N_JUNK > 0:
        junk = pd.DataFrame(rng.normal(size=(len(Xb), N_JUNK)),
                            columns=[f"junk_{j}" for j in range(N_JUNK)])
        Xb = pd.concat([Xb, junk], axis=1)
    yb = pd.Series(yb)
    sel = SFS(LogisticRegression(max_iter=1000), k_features=K_TARGET,
              forward=True, floating=False, scoring="accuracy", cv=0)
    sel.fit(Xb, yb)
    ins.append(sel.subsets_[K_TARGET]["avg_score"])
    cols = list(sel.subsets_[K_TARGET]["feature_names"])
    Xtr, Xte, ytr, yte = train_test_split(Xb[cols], yb, test_size=0.3,
                                          random_state=int(rng.integers(1e9)), stratify=yb)
    outs.append(LogisticRegression(max_iter=1000).fit(Xtr, ytr).score(Xte, yte))

print(f"in-sample  mean={np.mean(ins):.4f}  sd={np.std(ins):.4f}  {np.round(ins,4)}")
print(f"hold-out   mean={np.mean(outs):.4f}  sd={np.std(outs):.4f}  {np.round(outs,4)}")
print("try: N_JUNK=6 (wrappers should ignore junk), FLIP_P=0.10 (both scores drop), N_SUB=300 (variance up)")


## 10. Audience rewrite (worked)

**Expert (epidemiologist / statistician).**  
Binary obesity indicator, n = 2,111, prevalence 46%. Unpenalized LR on 18 encoded items has resubstitution accuracy 0.766. SFS with `cv=0` reaches 0.784 at 9 items and already matches the full card at 6. The five-item core {Age, family history, FAVC, CAEC, SCC} is stable across SFS / SBS / RFE. These are *in-sample* figures; a 70/30 split shrinks the SFS edge. Transport dummies are a simplex — drop a reference level before interpreting coefficients. Not a causal model and not a diagnostic device.

**Technician (survey / data analyst).**  
Pipeline: `LogisticRegression(max_iter=1000)` → `SFS(..., k_features=9, forward=True, floating=False, cv=0)` → `plot_sfs`. Keep Age, family_history_with_overweight, FAVC, CAEC, SCC as the short form. Use `RFE` only after `StandardScaler`. For a production card switch `cv=0` to `cv=5` and lock the column list on a held-out year. Code lives in `WrapFS_Obes.py`.

**Executive (public-health director).**  
A 9-question version of the eating-habits survey classifies obesity *at least as well* as the 18-question version in this teaching file (78% vs 77% on the same rows). The items that matter are family history, frequent high-calorie food, snacking between meals, whether the person already tracks calories, and age. That is a shorter instrument, not a clinic-ready test. Next step: validate on a new sample before cutting questions from a live programme.

**Nonspecialist.**  
We asked 2,111 people about meals, exercise, and how they get around, then tried to tell who is obese from the answers. Using every question we were right about 77% of the time. Using nine of the questions we were right about 78% of the time. The useful questions are mostly “does obesity run in the family?”, “do you often eat high-calorie food?”, “do you snack?”, “do you watch calories?”, and age. This is a class exercise, not a medical test.

**Can / cannot.**  
Can: shorten a questionnaire; teach wrapper vs filter vs L1.  
Cannot: diagnose a person; replace BMI; claim that cutting snacks *causes* lower obesity.


## 11. Takeaways

1. Wrappers wrap a *specific* estimator — here logistic regression. A different model can pick a different card.
2. SFS-9 **beats** the 18-feature model *in-sample*; SBS-7 does not. Greedy search is not a theorem.
3. RFE needs scaling; SFS/SBS on raw units still work because they score accuracy, not coefficient size.
4. `cv=0` overstates performance. Always report a hold-out or a CV mean before you retire questions.
5. Family history + high-calorie food + snacking + calorie monitoring + age is the stable core on this file.
